# Stage 3: Transformer Modeling for AutoTuneX

This notebook implements a Time-Series Transformer model for autoscaling prediction:
- Sliding window data preparation
- Multi-head attention Transformer architecture
- Training with early stopping
- Comparison with baseline models

**Target**: Predict next-step request_rate and latency_p95 using temporal patterns

## Import required libraries and set random seeds for reproducibility

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import json
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU'))} devices")
print("Libraries loaded successfully")

## Load prediction-ready dataset and prepare for time-series modeling

In [ ]:
df = pd.read_csv('/kaggle/input/autotunex-data/prediction_ready_dataset.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

## Separate features and targets, then create time-series splits

In [ ]:
target_cols = ['target_request_rate', 'target_latency_p95']
feature_cols = [col for col in df.columns if col not in target_cols + ['target_replica_count', 'timestamp']]

X = df[feature_cols].values
y = df[target_cols].values

# Time-series split (70/15/15)
n_samples = len(X)
train_size = int(0.70 * n_samples)
val_size = int(0.15 * n_samples)

X_train_full = X[:train_size]
y_train_full = y[:train_size]
X_val_full = X[train_size:train_size + val_size]
y_val_full = y[train_size:train_size + val_size]
X_test_full = X[train_size + val_size:]
y_test_full = y[train_size + val_size:]

print(f"Features: {len(feature_cols)}")
print(f"Targets: {len(target_cols)}")
print(f"\nTrain samples: {len(X_train_full)} (70%)")
print(f"Val samples: {len(X_val_full)} (15%)")
print(f"Test samples: {len(X_test_full)} (15%)")

## Normalize features using StandardScaler (fit on training data only)

In [ ]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train_full)
X_val_scaled = scaler_X.transform(X_val_full)
X_test_scaled = scaler_X.transform(X_test_full)

y_train_scaled = scaler_y.fit_transform(y_train_full)
y_val_scaled = scaler_y.transform(y_val_full)
y_test_scaled = scaler_y.transform(y_test_full)

print("Feature scaling complete")
print(f"Train X shape: {X_train_scaled.shape}")
print(f"Train y shape: {y_train_scaled.shape}")

## Task 3.1: Create sliding window sequences for time-series Transformer input

In [ ]:
def create_sequences(X, y, window_size=10):
    """
    Create sliding window sequences for time-series prediction.
    
    Args:
        X: Input features (n_samples, n_features)
        y: Target values (n_samples, n_targets)
        window_size: Number of past timesteps to use
    
    Returns:
        X_seq: Sequences of shape (n_sequences, window_size, n_features)
        y_seq: Targets of shape (n_sequences, n_targets)
    """
    X_seq, y_seq = [], []
    
    for i in range(len(X) - window_size):
        X_seq.append(X[i:i + window_size])
        y_seq.append(y[i + window_size])
    
    return np.array(X_seq), np.array(y_seq)

# Create sequences with window size of 10
WINDOW_SIZE = 10

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, WINDOW_SIZE)
X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val_scaled, WINDOW_SIZE)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, WINDOW_SIZE)

print(f"Window size: {WINDOW_SIZE} timesteps")
print(f"\nSequence shapes:")
print(f"Train X: {X_train_seq.shape} (samples, window, features)")
print(f"Train y: {y_train_seq.shape} (samples, targets)")
print(f"Val X: {X_val_seq.shape}")
print(f"Test X: {X_test_seq.shape}")

## Task 3.2: Define Transformer block with multi-head attention and feed-forward network

In [ ]:
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    """
    Transformer encoder block with multi-head attention.
    
    Args:
        inputs: Input tensor
        head_size: Dimension of attention heads
        num_heads: Number of attention heads
        ff_dim: Hidden layer size in feed-forward network
        dropout: Dropout rate
    
    Returns:
        Output tensor after transformer block
    """
    # Multi-head attention
    attention_output = layers.MultiHeadAttention(
        key_dim=head_size, 
        num_heads=num_heads, 
        dropout=dropout
    )(inputs, inputs)
    attention_output = layers.Dropout(dropout)(attention_output)
    attention_output = layers.LayerNormalization(epsilon=1e-6)(inputs + attention_output)
    
    # Feed-forward network
    ff_output = layers.Conv1D(filters=ff_dim, kernel_size=1, activation='relu')(attention_output)
    ff_output = layers.Dropout(dropout)(ff_output)
    ff_output = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(ff_output)
    ff_output = layers.Dropout(dropout)(ff_output)
    
    # Residual connection and normalization
    output = layers.LayerNormalization(epsilon=1e-6)(attention_output + ff_output)
    
    return output

print("Transformer encoder function defined")

## Build complete Transformer model architecture with positional encoding

In [ ]:
def build_transformer_model(
    input_shape,
    num_outputs,
    head_size=64,
    num_heads=4,
    ff_dim=128,
    num_transformer_blocks=2,
    mlp_units=[128],
    dropout=0.2,
    mlp_dropout=0.3
):
    """
    Build complete Transformer model for time-series prediction.
    
    Args:
        input_shape: (window_size, n_features)
        num_outputs: Number of target variables
        head_size: Dimension of each attention head
        num_heads: Number of attention heads
        ff_dim: Feed-forward network hidden dimension
        num_transformer_blocks: Number of Transformer encoder blocks
        mlp_units: Dense layers after Transformer
        dropout: Dropout rate in Transformer
        mlp_dropout: Dropout rate in MLP
    
    Returns:
        Compiled Keras model
    """
    inputs = layers.Input(shape=input_shape)
    x = inputs
    
    # Add positional encoding
    positions = tf.range(start=0, limit=input_shape[0], delta=1)
    position_embedding = layers.Embedding(
        input_dim=input_shape[0], output_dim=input_shape[1]
    )(positions)
    x = x + position_embedding
    
    # Stack Transformer encoder blocks
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
    
    # Global average pooling to get fixed-size representation
    x = layers.GlobalAveragePooling1D()(x)
    
    # MLP for final prediction
    for dim in mlp_units:
        x = layers.Dense(dim, activation='relu')(x)
        x = layers.Dropout(mlp_dropout)(x)
    
    # Output layer
    outputs = layers.Dense(num_outputs)(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    
    return model

# Build model
input_shape = (WINDOW_SIZE, X_train_seq.shape[2])
num_outputs = y_train_seq.shape[1]

model = build_transformer_model(
    input_shape=input_shape,
    num_outputs=num_outputs,
    head_size=64,
    num_heads=4,
    ff_dim=128,
    num_transformer_blocks=2,
    mlp_units=[128, 64],
    dropout=0.2,
    mlp_dropout=0.3
)

print("Transformer model built successfully")
print(f"\nModel architecture:")
model.summary()

## Compile model with Adam optimizer and MSE loss

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print("Model compiled with Adam optimizer and MSE loss")

## Task 3.3: Train Transformer model with early stopping and model checkpointing

In [ ]:
# Define callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

model_checkpoint = keras.callbacks.ModelCheckpoint(
    'best_transformer_model.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

print("Training callbacks configured:")
print("  - Early stopping (patience=10)")
print("  - Learning rate reduction (patience=5)")
print("  - Model checkpointing")
print("\nStarting training...\n")

# Train model
history = model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr, model_checkpoint],
    verbose=1
)

print("\nTraining complete!")

## Visualize training history (loss and MAE curves)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE curve
axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Val MAE', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Training and Validation MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('transformer_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Training completed in {len(history.history['loss'])} epochs")
print(f"Best validation loss: {min(history.history['val_loss']):.6f}")

## Task 3.4: Evaluate Transformer on test set and denormalize predictions

In [ ]:
# Make predictions on all sets
y_train_pred_scaled = model.predict(X_train_seq)
y_val_pred_scaled = model.predict(X_val_seq)
y_test_pred_scaled = model.predict(X_test_seq)

# Denormalize predictions and true values
y_train_pred = scaler_y.inverse_transform(y_train_pred_scaled)
y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled)
y_test_pred = scaler_y.inverse_transform(y_test_pred_scaled)

y_train_true = scaler_y.inverse_transform(y_train_seq)
y_val_true = scaler_y.inverse_transform(y_val_seq)
y_test_true = scaler_y.inverse_transform(y_test_seq)

print("Predictions generated and denormalized")
print(f"Test predictions shape: {y_test_pred.shape}")

## Calculate evaluation metrics (MAE, RMSE, R²) for both targets

In [ ]:
def evaluate_predictions(y_true, y_pred, target_names, set_name):
    """
    Calculate and display evaluation metrics.
    """
    results = {}
    
    print(f"\n=== {set_name} Set Results ===")
    for i, target in enumerate(target_names):
        mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
        r2 = r2_score(y_true[:, i], y_pred[:, i])
        
        results[target] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
        
        print(f"\n{target}:")
        print(f"  MAE:  {mae:.4f}")
        print(f"  RMSE: {rmse:.4f}")
        print(f"  R²:   {r2:.4f}")
    
    return results

# Evaluate on all sets
transformer_train_results = evaluate_predictions(y_train_true, y_train_pred, target_cols, "Train")
transformer_val_results = evaluate_predictions(y_val_true, y_val_pred, target_cols, "Validation")
transformer_test_results = evaluate_predictions(y_test_true, y_test_pred, target_cols, "Test")

## Calculate spike prediction accuracy for request rate

In [ ]:
def calculate_spike_accuracy(y_true, y_pred, threshold_percentile=95):
    """
    Calculate accuracy of predicting traffic spikes.
    """
    threshold = np.percentile(y_true[:, 0], threshold_percentile)
    true_spikes = y_true[:, 0] > threshold
    pred_spikes = y_pred[:, 0] > threshold
    accuracy = np.mean(true_spikes == pred_spikes)
    return accuracy, threshold

spike_acc_test, spike_threshold = calculate_spike_accuracy(y_test_true, y_test_pred)

print(f"\n=== Spike Prediction Analysis ===")
print(f"Threshold (95th percentile): {spike_threshold:.2f}")
print(f"Spike prediction accuracy: {spike_acc_test:.4f} ({spike_acc_test*100:.2f}%)")

## Load baseline results for comparison with Transformer

In [ ]:
# Load baseline results from Stage 2
try:
    baseline_rr = pd.read_csv('/kaggle/input/autotunex-baseline/baseline_results_request_rate.csv')
    baseline_lat = pd.read_csv('/kaggle/input/autotunex-baseline/baseline_results_latency_p95.csv')
    
    print("Baseline results loaded successfully")
    print("\nRequest Rate Baseline Results:")
    print(baseline_rr[['Model', 'Test MAE', 'Test RMSE', 'Test R²']].to_string(index=False))
    print("\nLatency P95 Baseline Results:")
    print(baseline_lat[['Model', 'Test MAE', 'Test RMSE', 'Test R²']].to_string(index=False))
except:
    print("Baseline results not found. Creating placeholder data.")
    # Placeholder baseline results from Stage 2
    baseline_rr = pd.DataFrame({
        'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
        'Test MAE': [3060.17, 3437.46, 4911.10],
        'Test RMSE': [4464.78, 4910.95, 7093.66],
        'Test R²': [0.8672, 0.8393, 0.6647],
        'Spike Accuracy': [0.9573, 0.9498, 0.9498]
    })
    
    baseline_lat = pd.DataFrame({
        'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
        'Test MAE': [0.0394, 0.0721, 0.0875],
        'Test RMSE': [0.0522, 0.0817, 0.1149],
        'Test R²': [0.6894, 0.2387, -0.5055]
    })

## Create comprehensive comparison table: Transformer vs Baseline models

In [ ]:
# Add Transformer results to baseline comparison
transformer_rr_row = pd.DataFrame({
    'Model': ['Transformer'],
    'Test MAE': [transformer_test_results['target_request_rate']['MAE']],
    'Test RMSE': [transformer_test_results['target_request_rate']['RMSE']],
    'Test R²': [transformer_test_results['target_request_rate']['R2']],
    'Spike Accuracy': [spike_acc_test]
})

transformer_lat_row = pd.DataFrame({
    'Model': ['Transformer'],
    'Test MAE': [transformer_test_results['target_latency_p95']['MAE']],
    'Test RMSE': [transformer_test_results['target_latency_p95']['RMSE']],
    'Test R²': [transformer_test_results['target_latency_p95']['R2']]
})

# Combine with baselines
comparison_rr = pd.concat([baseline_rr, transformer_rr_row], ignore_index=True)
comparison_lat = pd.concat([baseline_lat, transformer_lat_row], ignore_index=True)

print("\n" + "="*80)
print("REQUEST RATE PREDICTION - MODEL COMPARISON")
print("="*80)
print(comparison_rr.to_string(index=False))

print("\n" + "="*80)
print("LATENCY P95 PREDICTION - MODEL COMPARISON")
print("="*80)
print(comparison_lat.to_string(index=False))

# Save comparison results
comparison_rr.to_csv('transformer_comparison_request_rate.csv', index=False)
comparison_lat.to_csv('transformer_comparison_latency_p95.csv', index=False)

print("\nComparison results saved!")

## Visualize model comparison with bar charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Request Rate R²
axes[0, 0].bar(comparison_rr['Model'], comparison_rr['Test R²'], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[0, 0].set_ylabel('R² Score')
axes[0, 0].set_title('Request Rate Prediction - R² Comparison')
axes[0, 0].set_ylim([0, 1])
axes[0, 0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(comparison_rr['Test R²']):
    axes[0, 0].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

# Request Rate MAE
axes[0, 1].bar(comparison_rr['Model'], comparison_rr['Test MAE'], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[0, 1].set_ylabel('MAE')
axes[0, 1].set_title('Request Rate Prediction - MAE Comparison (Lower is Better)')
axes[0, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(comparison_rr['Test MAE']):
    axes[0, 1].text(i, v + 100, f'{v:.1f}', ha='center', fontweight='bold')

# Latency P95 R²
axes[1, 0].bar(comparison_lat['Model'], comparison_lat['Test R²'], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[1, 0].set_ylabel('R² Score')
axes[1, 0].set_title('Latency P95 Prediction - R² Comparison')
axes[1, 0].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1, 0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(comparison_lat['Test R²']):
    axes[1, 0].text(i, v + 0.05 if v > 0 else v - 0.1, f'{v:.4f}', ha='center', fontweight='bold')

# Latency P95 MAE
axes[1, 1].bar(comparison_lat['Model'], comparison_lat['Test MAE'], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[1, 1].set_ylabel('MAE (seconds)')
axes[1, 1].set_title('Latency P95 Prediction - MAE Comparison (Lower is Better)')
axes[1, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(comparison_lat['Test MAE']):
    axes[1, 1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('transformer_vs_baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Comparison visualization saved!")

## Visualize Transformer predictions vs true values (scatter plots)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Request Rate
axes[0].scatter(y_test_true[:, 0], y_test_pred[:, 0], alpha=0.3, s=10)
axes[0].plot([y_test_true[:, 0].min(), y_test_true[:, 0].max()],
             [y_test_true[:, 0].min(), y_test_true[:, 0].max()],
             'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('True Request Rate')
axes[0].set_ylabel('Predicted Request Rate')
axes[0].set_title(f'Request Rate Prediction\nR² = {transformer_test_results["target_request_rate"]["R2"]:.4f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Latency P95
axes[1].scatter(y_test_true[:, 1], y_test_pred[:, 1], alpha=0.3, s=10, color='orange')
axes[1].plot([y_test_true[:, 1].min(), y_test_true[:, 1].max()],
             [y_test_true[:, 1].min(), y_test_true[:, 1].max()],
             'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('True Latency P95 (seconds)')
axes[1].set_ylabel('Predicted Latency P95 (seconds)')
axes[1].set_title(f'Latency P95 Prediction\nR² = {transformer_test_results["target_latency_p95"]["R2"]:.4f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('transformer_predictions_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print("Scatter plot visualization saved!")

## Visualize time-series predictions over a sample window

In [ ]:
sample_size = 200
sample_indices = range(sample_size)

fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# Request Rate
axes[0].plot(sample_indices, y_test_true[:sample_size, 0], 'k-', label='True', linewidth=2)
axes[0].plot(sample_indices, y_test_pred[:sample_size, 0], 'r--', label='Transformer', alpha=0.7, linewidth=2)
axes[0].set_xlabel('Sample Index')
axes[0].set_ylabel('Request Rate')
axes[0].set_title('Request Rate Prediction - Time Series Comparison')
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)

# Latency P95
axes[1].plot(sample_indices, y_test_true[:sample_size, 1], 'k-', label='True', linewidth=2)
axes[1].plot(sample_indices, y_test_pred[:sample_size, 1], 'r--', label='Transformer', alpha=0.7, linewidth=2)
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Latency P95 (seconds)')
axes[1].set_title('Latency P95 Prediction - Time Series Comparison')
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('transformer_timeseries_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print("Time series visualization saved!")

## Generate final summary report with key findings

In [ ]:
# Determine best models
best_model_rr = comparison_rr.loc[comparison_rr['Test R²'].idxmax(), 'Model']
best_r2_rr = comparison_rr['Test R²'].max()

best_model_lat = comparison_lat.loc[comparison_lat['Test R²'].idxmax(), 'Model']
best_r2_lat = comparison_lat['Test R²'].max()

# Calculate improvement over best baseline
best_baseline_rr_r2 = baseline_rr['Test R²'].max()
improvement_rr = ((transformer_test_results['target_request_rate']['R2'] - best_baseline_rr_r2) / best_baseline_rr_r2) * 100

best_baseline_lat_r2 = baseline_lat['Test R²'].max()
improvement_lat = ((transformer_test_results['target_latency_p95']['R2'] - best_baseline_lat_r2) / best_baseline_lat_r2) * 100

print("\n" + "="*80)
print("STAGE 3: TRANSFORMER MODELING - FINAL SUMMARY")
print("="*80)

print("\n📊 MODEL PERFORMANCE")
print("-" * 80)
print(f"\nRequest Rate Prediction:")
print(f"  Transformer R²: {transformer_test_results['target_request_rate']['R2']:.4f}")
print(f"  Best Baseline R²: {best_baseline_rr_r2:.4f} (Linear Regression)")
print(f"  Improvement: {improvement_rr:+.2f}%")
print(f"  Spike Detection Accuracy: {spike_acc_test:.4f}")

print(f"\nLatency P95 Prediction:")
print(f"  Transformer R²: {transformer_test_results['target_latency_p95']['R2']:.4f}")
print(f"  Best Baseline R²: {best_baseline_lat_r2:.4f} (Linear Regression)")
print(f"  Improvement: {improvement_lat:+.2f}%")

print("\n🏆 BEST MODELS")
print("-" * 80)
print(f"Request Rate: {best_model_rr} (R² = {best_r2_rr:.4f})")
print(f"Latency P95: {best_model_lat} (R² = {best_r2_lat:.4f})")

print("\n✅ KEY ACHIEVEMENTS")
print("-" * 80)
print("  ✓ Transformer architecture implemented with multi-head attention")
print("  ✓ Sliding window sequences created for temporal modeling")
print("  ✓ Model trained with early stopping and learning rate scheduling")
print("  ✓ Comprehensive comparison with baseline models completed")
print(f"  ✓ {'Transformer outperforms baselines' if improvement_rr > 0 and improvement_lat > 0 else 'Mixed results compared to baselines'}")

print("\n📁 FILES GENERATED")
print("-" * 80)
print("  - best_transformer_model.h5 (trained model weights)")
print("  - transformer_training_history.png (loss curves)")
print("  - transformer_vs_baseline_comparison.png (bar charts)")
print("  - transformer_predictions_scatter.png (scatter plots)")
print("  - transformer_timeseries_predictions.png (time series)")
print("  - transformer_comparison_request_rate.csv")
print("  - transformer_comparison_latency_p95.csv")

print("\n" + "="*80)
print("STAGE 3 COMPLETE - READY FOR STAGE 4 (MULTI-OBJECTIVE DECISION ENGINE)")
print("="*80)

## Save metadata and results for documentation

In [ ]:
metadata = {
    'stage': 'Stage 3: Transformer Modeling',
    'model_architecture': {
        'type': 'Time-Series Transformer',
        'window_size': WINDOW_SIZE,
        'num_heads': 4,
        'num_transformer_blocks': 2,
        'ff_dim': 128,
        'mlp_units': [128, 64],
        'dropout': 0.2,
        'total_params': model.count_params()
    },
    'training': {
        'epochs_trained': len(history.history['loss']),
        'batch_size': 32,
        'optimizer': 'Adam',
        'learning_rate': 0.001,
        'best_val_loss': float(min(history.history['val_loss']))
    },
    'test_performance': {
        'request_rate': {
            'mae': float(transformer_test_results['target_request_rate']['MAE']),
            'rmse': float(transformer_test_results['target_request_rate']['RMSE']),
            'r2': float(transformer_test_results['target_request_rate']['R2'])
        },
        'latency_p95': {
            'mae': float(transformer_test_results['target_latency_p95']['MAE']),
            'rmse': float(transformer_test_results['target_latency_p95']['RMSE']),
            'r2': float(transformer_test_results['target_latency_p95']['R2'])
        },
        'spike_accuracy': float(spike_acc_test)
    },
    'improvement_vs_baseline': {
        'request_rate_improvement_pct': float(improvement_rr),
        'latency_p95_improvement_pct': float(improvement_lat)
    },
    'best_models': {
        'request_rate': best_model_rr,
        'latency_p95': best_model_lat
    }
}

with open('transformer_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved to transformer_metadata.json")
print("\n🎉 Stage 3: Transformer Modeling Complete! 🎉")